In [1]:
import pandas as pd

In [2]:
wiki = pd.read_parquet("C:/Users/luisc/local/projetos/videos/rnn_corrector/data/wikipedia_2025.pq")
wiki

,id,revid:,title,url,text
0,5674408,662262,Anna Chandy,https://pt.wikipedia.org/wiki?curid=5674408,"Anna Chandy (1905-1996), também conhecida como..."
1,5674410,321756,Ajahn Mahā Bua,https://pt.wikipedia.org/wiki?curid=5674410,"Ajahn Mahā Bua (Udon Thani, 12 de agosto de 19..."
2,5674413,2702392,Debra Elmegreen,https://pt.wikipedia.org/wiki?curid=5674413,"Debra Meloy Elmegreen (South Bend, Indiana, 23..."
3,5674416,2123556,Laura Ferrarese,https://pt.wikipedia.org/wiki?curid=5674416,Laura Ferrarese é uma pesquisadora em ciência ...
4,5674418,710641,Anna Chandi,https://pt.wikipedia.org/wiki?curid=5674418,
...,...,...,...,...,...
1949914,302397,1082486,The Grifters,https://pt.wikipedia.org/wiki?curid=302397,"The Grifters (""br"": Os Imorais; ""pt"": Anatomia..."
1949915,302398,4468,Calpaínas,https://pt.wikipedia.org/wiki?curid=302398,
1949916,302399,662262,Crystal Planet,https://pt.wikipedia.org/wiki?curid=302399,Crystal Planet é um álbum do guitarrista de ro...
1949917,302400,3288074,Cláudio Paiva,https://pt.wikipedia.org/wiki?curid=302400,"Cláudio Paiva (Niterói, 11 de outubro de 1957)..."


In [3]:
wiki = wiki[["text"]]
wiki = wiki[wiki.text.str.len() > 10].reset_index(drop=True)
wiki

,text
0,"Anna Chandy (1905-1996), também conhecida como..."
1,"Ajahn Mahā Bua (Udon Thani, 12 de agosto de 19..."
2,"Debra Meloy Elmegreen (South Bend, Indiana, 23..."
3,Laura Ferrarese é uma pesquisadora em ciência ...
4,Ellen Dorrit Hoffleit (12 de março de 1907 – 9...
...,...
1129081,The Extremist é o quarto álbum do guitarrista ...
1129082,"The Grifters (""br"": Os Imorais; ""pt"": Anatomia..."
1129083,Crystal Planet é um álbum do guitarrista de ro...
1129084,"Cláudio Paiva (Niterói, 11 de outubro de 1957)..."


In [4]:
simples = pd.read_parquet("data/base_treino_simples.pq")
simples

,texto,len,split
0,Sempre gosto de comprar no Shoptime e adoro ok.,47,valid
1,O clichê mais tapa na cara que já se viu.,41,valid
2,Eu ñ seiinstalei ainda,22,valid
3,entre as comédias francesas mais engraçadas qu...,123,valid
4,Que filme ruim. Meu Deus.,25,valid
...,...,...,...
1897280,Ótimo este Clean Master!!!,26,train
1897281,Bom mais poderia melhorar,25,train
1897282,"produto sem qualidade , as peças não encaixam ...",118,train
1897283,Isto é uma oportunidade para apanharem dinheir...,73,train


In [5]:
df = pd.concat([
    wiki.rename(columns={"text": "texto"}),
    simples
], axis=0).reset_index(drop=True)
df

,texto,len,split
0,"Anna Chandy (1905-1996), também conhecida como...",NaN,NaN
1,"Ajahn Mahā Bua (Udon Thani, 12 de agosto de 19...",NaN,NaN
2,"Debra Meloy Elmegreen (South Bend, Indiana, 23...",NaN,NaN
3,Laura Ferrarese é uma pesquisadora em ciência ...,NaN,NaN
4,Ellen Dorrit Hoffleit (12 de março de 1907 – 9...,NaN,NaN
...,...,...,...
3026366,Ótimo este Clean Master!!!,26.0,train
3026367,Bom mais poderia melhorar,25.0,train
3026368,"produto sem qualidade , as peças não encaixam ...",118.0,train
3026369,Isto é uma oportunidade para apanharem dinheir...,73.0,train


In [7]:
df["len"] = df.texto.str.len()
df

,texto,len,split
0,"Anna Chandy (1905-1996), também conhecida como...",1475,NaN
1,"Ajahn Mahā Bua (Udon Thani, 12 de agosto de 19...",3379,NaN
2,"Debra Meloy Elmegreen (South Bend, Indiana, 23...",1819,NaN
3,Laura Ferrarese é uma pesquisadora em ciência ...,751,NaN
4,Ellen Dorrit Hoffleit (12 de março de 1907 – 9...,2394,NaN
...,...,...,...
3026366,Ótimo este Clean Master!!!,26,train
3026367,Bom mais poderia melhorar,25,train
3026368,"produto sem qualidade , as peças não encaixam ...",118,train
3026369,Isto é uma oportunidade para apanharem dinheir...,73,train


In [8]:
import re
import pandas as pd

def check_text_quality(text, min_words=5, max_symbol_ratio=0.1, max_digit_ratio=0.1):
    """
    Avalia a qualidade do texto com base em regras manuais
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return False
        
    # 1. Filtro de comprimento mínimo (evita textos muito curtos)
    words = text.split()
    if len(words) < min_words:
        return False
        
    # 2. Razão de símbolos (pontuação e caracteres especiais)
    # Textos com excesso de símbolos (como ".......") são ruído 
    symbols = len(re.findall(r'[^\w\s]', text))
    symbol_ratio = symbols / len(text)
    if symbol_ratio > max_symbol_ratio:
        return False
        
    # 3. Razão de dígitos (evita tabelas, códigos numéricos ou datas excessivas)
    digits = len(re.findall(r'\d', text))
    digit_ratio = digits / len(text)
    if digit_ratio > max_digit_ratio:
        return False
        
    # 4. Verificação de repetição de caracteres (o problema do "....")
    # Se um caractere se repete mais de 4 vezes seguidas, é provável que seja ruído
    if re.search(r'(.)\1{4,}', text):
        return False
        
    # 5. Verificação de repetição de palavras (BoW repetitivo)
    if len(words) > 10:
        unique_words_ratio = len(set(words)) / len(words)
        if unique_words_ratio < 0.3: # Ex: "o o o o o o"
            return False

    return True

In [9]:
df["filtra"] = df.texto.apply(lambda x: int(check_text_quality(x)))
df.filtra.value_counts()

filtra
1    2636087
0     390284
Name: count, dtype: int64

In [13]:
print(df[df.filtra == 0].sample(1).texto.values[0])

Simplesmente: foda!


In [14]:
df = df[df.filtra == 1].reset_index(drop=True).drop("filtra", axis=1)
df

,texto,len,split
0,"Anna Chandy (1905-1996), também conhecida como...",1475,NaN
1,"Ajahn Mahā Bua (Udon Thani, 12 de agosto de 19...",3379,NaN
2,"Debra Meloy Elmegreen (South Bend, Indiana, 23...",1819,NaN
3,Laura Ferrarese é uma pesquisadora em ciência ...,751,NaN
4,Ellen Dorrit Hoffleit (12 de março de 1907 – 9...,2394,NaN
...,...,...,...
2636082,Gostei do filme. Vai além de bons gráficos e c...,163,train
2636083,"Muda MUITA coisa em relação aos mais antigos, ...",126,train
2636084,"produto sem qualidade , as peças não encaixam ...",118,train
2636085,Isto é uma oportunidade para apanharem dinheir...,73,train


In [15]:
split = ["valid" for _ in range(500)] + ["train" for _ in range(len(df) - 500)]

df = df.sample(frac=1.0, replace=False).reset_index(drop=True)
df["split"] = split
df

,texto,len,split
0,Jackson's Arm é uma pequena cidade localizada ...,183,valid
1,"Muito bom, criativo. E Kate, como sempre, acim...",59,valid
2,"Legal, mas poderia ser melhor, não mostra muit...",162,valid
3,Realmente muito bom. Não achei tão cansativo q...,73,valid
4,Revi hoje o filme e mesmo com poucas cenas de...,108,valid
...,...,...,...
2636082,"As asas do meu ideal, a glória do meu Brasil!",45,train
2636083,Expectativa gera frustração.... esperei demais...,93,train
2636084,Werewolf ( ou O Lobisomem Ataca Outra Vez) é u...,1537,train
2636085,A Tempestade tropical Ingrid foi um ciclone tr...,2953,train


In [16]:
df.groupby("split")["len"].sum()

split
train    2394443423
valid        546631
Name: len, dtype: int64

In [ ]:
df.to_parquet("data/base_treino_wiki_reviews.pq")